### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. I know that parrots are the only birds that can mimic human speech, but why is that? Let me start by recalling what I know. Parrots are part of the Psittaciformes order, right? They have strong beaks and zygodactyl feet, which means two toes pointing forward and two backward. That might help them climb trees, but how does that relate to talking?\n\nI remember reading that parrots have a special vocal organ. Maybe it\'s called the syrinx? I think the syrinx is the vocal organ in birds, and it\'s different from the human larynx. How does that work in parrots compared to other birds? If other birds can\'t talk, maybe their syrinx isn\'t structured the same way. So the syrinx in parrots allows them to produce a wide range of sounds, including human words. But why do they actually talk? Is it for communication, or do they just mimic sounds?\n\nAnother thought: maybe talking is a way for parrots to bond with their hu

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': '91ztgx9qr', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.149864856, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006571559, 'prompt_tokens_details': None, 'queue_time': 0.35544973, 'total_time': 0.156436415}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f1eeb-5cc9-7441-98

### Tool Execution Loops

In [5]:
messages = [{"role": "user", "content": "What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny. A perfect day to enjoy outdoor activities! 😊


In [6]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are provided, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'gayp0bvkp', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 153, 'total_tokens': 257, 'completion_time': 0.173526364, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.006020531, 'prompt_tokens_details': None, 'queue_time': 0.066477919, 'total_time': 0.179546895}, 'model_name': 'qwen/qwen3-32b', 'syst

In [7]:
ai_msg

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are provided, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'gayp0bvkp', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 153, 'total_tokens': 257, 'completion_time': 0.173526364, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.006020531, 'prompt_tokens_details': None, 'queue_time': 0.066477919, 'total_time': 0.179546895}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'f

In [8]:
ai_msg.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Boston'},
  'id': 'gayp0bvkp',
  'type': 'tool_call'}]

In [13]:
messages[-1]

ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='gayp0bvkp')